<a href="https://colab.research.google.com/github/vneumannufprbr/TrabajosenPython/blob/main/Dashboard_Interactivo_de_Anomal%C3%ADas_(Ultimo_mes)_Ok_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Celda 1: Instalación e Importación de Librerías
# ==============================================================================
# Instalar las librerías necesarias. Plotly es para gráficos interactivos.
!pip install openpyxl plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta

# ==============================================================================
# Celda 2: Carga y Preparación de Datos
# ==============================================================================
# --- Carga de Datos ---
# Por favor, sube tu archivo 'matriz_diagnostico_escalada_completa.xlsx' al entorno de Colab.
try:
    file_path = 'matriz_diagnostico_escalada_completa.xlsx'
    df = pd.read_excel(file_path)
    # Asegurarse de que la fecha sea el índice y esté en formato datetime
    if 'fecha' in df.columns:
        df['fecha'] = pd.to_datetime(df['fecha'])
    print("Archivo cargado y procesado exitosamente.")
    print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print(f"Error: El archivo '{file_path}' no se encontró.")
    print("Por favor, sube el archivo al explorador de archivos de Colab (panel izquierdo) y vuelve a ejecutar esta celda.")
except Exception as e:
    print(f"Ocurrió un error al leer el archivo: {e}")

# --- Limpieza y Preparación Adicional ---
# Asegurar que las columnas de texto no tengan espacios extra
df['etiqueta_padre'] = df['etiqueta_padre'].str.strip()
df['sigla_estructura'] = df['sigla_estructura'].str.strip()

# Definir la paleta de colores (AHORA CON CÓDIGOS HEX) y el orden para consistencia
color_map = {
    'Anomalía Estructural': '#FF0000', # Rojo
    'Anomalía Regional': '#FFA500',   # Naranja
    'Sensor Defectuoso': '#0000FF',   # Azul
    'Sistema Normal': '#008000'      # Verde
}

# ==============================================================================
# Celda 3: Dashboard Nivel 1 - Panel de Control Estratégico
# ==============================================================================
print("=============================================")
print("  Panel de Control Estratégico (Nivel 1)")
print("=============================================")

# --- Cálculo de KPIs ---
today = df['fecha'].max() # Usar la última fecha de los datos como referencia
last_7_days = today - timedelta(days=7)
last_30_days = today - timedelta(days=30)
prev_30_days_start = last_30_days - timedelta(days=30)

# KPI 1: Alertas Críticas (Últimos 7 días)
critical_alerts_df = df[(df['fecha'] >= last_7_days) & (df['diagnostic_category'].isin(['Anomalía Estructural', 'Anomalía Regional']))]
kpi1_value = len(critical_alerts_df)
print(f"\nKPI 1: Alertas Críticas (Últimos 7 días) -> {kpi1_value}")

# KPI 2: Sensores en estado "Defectuoso" (Último mes)
defective_sensors_df = df[(df['fecha'] >= last_30_days) & (df['diagnostic_category'] == 'Sensor Defectuoso')]
kpi2_value = defective_sensors_df['nombre_etiqueta'].nunique()
print(f"KPI 2: Sensores únicos en estado 'Defectuoso' (Último mes) -> {kpi2_value}")

# KPI 3: Salud General del Sistema (Último mes)
health_df = df[df['fecha'] >= last_30_days]
kpi3_value = (health_df['diagnostic_category'] == 'Sistema Normal').sum() / len(health_df) * 100 if not health_df.empty else 100
print(f"KPI 3: Salud General del Sistema (Último mes) -> {kpi3_value:.2f}% Normal")

# --- Gráfico 1: Mapa Sinóptico (Estado por Estructura) ---
print("\n--- Mapa Sinóptico de la Presa (Estado por Estructura) ---")
# Determinar el estado más crítico por estructura en el último mes
status_df = df[df['fecha'] >= last_30_days]
category_severity = {'Anomalía Estructural': 4, 'Anomalía Regional': 3, 'Sensor Defectuoso': 2, 'Sistema Normal': 1}
status_df['severity'] = status_df['diagnostic_category'].map(category_severity)

structure_status = status_df.loc[status_df.groupby('sigla_estructura')['severity'].idxmax()]
# Mapear de vuelta a texto
severity_to_cat = {v: k for k, v in category_severity.items()}
structure_status['worst_category'] = structure_status['severity'].map(severity_to_cat)

# Crear la visualización del mapa sinóptico
fig_map = px.treemap(structure_status,
                     path=[px.Constant("Presa"), 'sigla_estructura'],
                     values='severity', # El tamaño no importa, solo la estructura
                     color='worst_category',
                     color_discrete_map=color_map, # Usar el mapa de colores hex
                     title='Estado Más Crítico por Estructura (Último Mes)')
fig_map.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig_map.show()

# --- Gráfico 2: Ranking de Grupos Críticos ---
print("\n--- Ranking de Grupos de Sensores con más Anomalías Críticas (Histórico) ---")
critical_df = df[df['diagnostic_category'].isin(['Anomalía Estructural', 'Anomalía Regional'])]
ranking = critical_df['etiqueta_padre'].value_counts().nlargest(10).sort_values()

fig_rank = px.bar(ranking,
                  x=ranking.values,
                  y=ranking.index,
                  orientation='h',
                  title='Top 10 Grupos con Más Anomalías Críticas (Estructurales y Regionales)',
                  labels={'x': 'Número de Alertas Críticas', 'y': 'Etiqueta Padre'},
                  color_discrete_sequence=['#c62828']) # Color rojo oscuro
fig_rank.update_layout(yaxis={'categoryorder':'total ascending'})
fig_rank.show()


# ==============================================================================
# Celda 4: Dashboard Nivel 2 - Vista de Investigación de Diagnóstico
# ==============================================================================
print("\n=============================================")
print("  Vista de Investigación de Diagnóstico (Nivel 2)")
print("=============================================")
print("Use los siguientes filtros para explorar los datos. La tabla y el gráfico se actualizarán.")

# --- Creación de Widgets Interactivos ---
style = {'description_width': 'initial'}
date_range_picker = widgets.DatePicker(description='Fecha Inicio:', style=style, value=df['fecha'].min().date())
category_dropdown = widgets.Dropdown(description='Categoría:', style=style, options=['Todas'] + list(color_map.keys()))
parent_label_dropdown = widgets.Dropdown(description='Etiqueta Padre:', style=style, options=['Todas'] + sorted(list(df['etiqueta_padre'].unique())))
update_button = widgets.Button(description='Aplicar Filtros')
output_area = widgets.Output()

# --- Función para actualizar el dashboard ---
def update_dashboard(b):
    with output_area:
        clear_output(wait=True)

        # 1. Filtrar datos
        start_date = pd.to_datetime(date_range_picker.value)

        filtered = df[df['fecha'] >= start_date]
        if category_dropdown.value != 'Todas':
            filtered = filtered[filtered['diagnostic_category'] == category_dropdown.value]
        if parent_label_dropdown.value != 'Todas':
            filtered = filtered[filtered['etiqueta_padre'] == parent_label_dropdown.value]

        print(f"Se encontraron {len(filtered)} alertas con los filtros aplicados.")

        # 2. Mostrar Tabla de Alertas
        if not filtered.empty:
            display_cols = ['fecha', 'nombre_etiqueta', 'etiqueta_padre', 'valor_promedio', 'diagnostic_category']
            table_data = filtered[display_cols].head(100)

            # --- LÍNEA CORREGIDA ---
            # Mapear categorías a códigos hex, usando gris (#808080) si no se encuentra
            hex_colors = table_data['diagnostic_category'].map(color_map).fillna('#808080')
            # Convertir hex a rgba para el fondo de la celda
            cell_colors = hex_colors.apply(lambda x: f'rgba({int(x[1:3], 16)}, {int(x[3:5], 16)}, {int(x[5:7], 16)}, 0.2)')
            # --- FIN DE LA CORRECCIÓN ---

            fig_table = go.Figure(data=[go.Table(
                header=dict(values=list(table_data.columns), fill_color='lightgrey', align='left'),
                cells=dict(values=[table_data[col].dt.strftime('%Y-%m-%d') if pd.api.types.is_datetime64_any_dtype(table_data[col]) else table_data[col] for col in table_data.columns],
                           fill_color=[cell_colors],
                           align='left'))
            ])
            fig_table.update_layout(title_text="Tabla de Alertas Filtradas (primeras 100)")
            display(fig_table)

            # 3. Gráfico de Serie Temporal y Contexto
            first_alert = table_data.iloc[0]
            sensor_to_plot = first_alert['nombre_etiqueta']
            alert_date = first_alert['fecha']

            sensor_full_history = df[df['nombre_etiqueta'] == sensor_to_plot]

            fig_ts = go.Figure()
            fig_ts.add_trace(go.Scatter(x=sensor_full_history['fecha'], y=sensor_full_history['valor_promedio'],
                                     mode='lines', name='Histórico del Sensor'))
            fig_ts.add_trace(go.Scatter(x=[alert_date], y=[first_alert['valor_promedio']],
                                     mode='markers', marker=dict(color='red', size=12, symbol='x'),
                                     name='Alerta Seleccionada'))

            fig_ts.update_layout(title=f"Serie Temporal para: {sensor_to_plot}",
                                 xaxis_title="Fecha", yaxis_title="Valor Promedio")
            display(fig_ts)

            # 4. Panel de Contexto
            context_info_rows = df[(df['fecha'] == alert_date) & (df['nombre_etiqueta'] == sensor_to_plot)]
            if not context_info_rows.empty:
                context_info = context_info_rows.iloc[0]
                print("\n--- Contexto de la Alerta Seleccionada ---")
                print(f"  Fecha: {alert_date.strftime('%Y-%m-%d')}")
                print(f"  Tipo de Origen: {context_info.get('tipo_origen_y', 'N/A')}")
                print(f"  Nivel del Embalse: {context_info.get('nivel_embalse', 'N/A')} m")
                print(f"  Local Geológico: {context_info.get('local_geologico', 'N/A')}")

        else:
            print("No se encontraron datos con los filtros seleccionados.")

# --- Conectar botón a la función y mostrar widgets ---
update_button.on_click(update_dashboard)

controls = widgets.VBox([
    widgets.HBox([date_range_picker, category_dropdown]),
    parent_label_dropdown,
    update_button
])

display(controls, output_area)

# Llamada inicial para poblar el dashboard
update_dashboard(None)

Archivo cargado y procesado exitosamente.
El dataset tiene 75566 filas y 32 columnas.
  Panel de Control Estratégico (Nivel 1)

KPI 1: Alertas Críticas (Últimos 7 días) -> 2
KPI 2: Sensores únicos en estado 'Defectuoso' (Último mes) -> 0
KPI 3: Salud General del Sistema (Último mes) -> 0.00% Normal

--- Mapa Sinóptico de la Presa (Estado por Estructura) ---


/tmp/ipython-input-134004471.py:81: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  status_df['severity'] = status_df['diagnostic_category'].map(category_severity)



--- Ranking de Grupos de Sensores con más Anomalías Críticas (Histórico) ---



  Vista de Investigación de Diagnóstico (Nivel 2)
Use los siguientes filtros para explorar los datos. La tabla y el gráfico se actualizarán.


Output()